<a href="https://colab.research.google.com/github/ivansuarez-AU/EGN321_Module2_Assignment2_2/blob/main/EGN321_Module2_Assignment2_2_Validated_Tool_Lab_COMPLETE_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EGN 321 — Module 2
## Assignment 2.2: Validated Engineering Tool — COMPLETE DEMO

This instructor demonstration notebook shows the complete workflow for Assignment 2.2:

1. Load the inherited workbook.
2. Trace the calculation chain.
3. Identify the duplicate-conversion defect.
4. Reuse a tested unit-conversion function.
5. Validate individual inputs.
6. Validate impossible combinations.
7. Rebuild the calculation with named intermediate values.
8. Verify known-correct workbook cases.
9. Test expected rejections.
10. Exercise boundary conditions.
11. Run a larger supplemental dataset.
12. Demonstrate how tests catch a reintroduced defect.

> **Teaching message:** A trustworthy engineering tool must know both **how to calculate** and **when to refuse**.


## 1. Upload the Assignment Files

For a live Colab demo, upload:

- `PUMP_HEAD_rev6.xlsx`
- optionally `PUMP_VALIDATION_DATA.xlsx`
- optionally the three CSV supplemental datasets

If you are running this notebook locally instead of Colab, place the files in the current working directory.


In [1]:
from pathlib import Path

try:
    from google.colab import files
    print("Running in Google Colab.")
    print("Upload PUMP_HEAD_rev6.xlsx and, optionally, PUMP_VALIDATION_DATA.xlsx / CSV files.")
    uploaded = files.upload()
except ImportError:
    print("Not running in Colab. Using files from the current working directory.")


Running in Google Colab.
Upload PUMP_HEAD_rev6.xlsx and, optionally, PUMP_VALIDATION_DATA.xlsx / CSV files.


Saving PUMP_HEAD_rev6.xlsx to PUMP_HEAD_rev6.xlsx


## 2. Install / Import Required Packages


In [2]:
import math
import csv
from pathlib import Path

try:
    from openpyxl import load_workbook
except ImportError:
    !pip -q install openpyxl
    from openpyxl import load_workbook

print("Environment ready.")


Environment ready.


## 3. Load the Inherited Workbook

The inherited workbook contains:

- external operating inputs,
- a legacy multi-step calculation,
- operating limits,
- independent reference cases.

The workbook is evidence to inspect — **not code to copy blindly**.


In [3]:
WORKBOOK = Path("PUMP_HEAD_rev6.xlsx")

if not WORKBOOK.exists():
    raise FileNotFoundError(
        "PUMP_HEAD_rev6.xlsx was not found. Upload it using Step 1, then rerun this cell."
    )

wb = load_workbook(WORKBOOK, data_only=False)
print("Sheets:", wb.sheetnames)


Sheets: ['Read Me', 'Inputs', 'Calculation Chain', 'Operating Limits', 'Reference Cases']


## 4. Inspect the External Inputs

Ask the class:

- Which values are external inputs?
- Which units enter the system?
- Which inputs will require conversion?
- Which values can remain in their current unit system?


In [4]:
ws = wb["Inputs"]

inputs = {}
for row in range(4, 10):
    name = ws[f"A{row}"].value
    value = ws[f"B{row}"].value
    unit = ws[f"C{row}"].value
    note = ws[f"D{row}"].value
    inputs[name] = value
    print(f"{name:24s} = {value:8.3f} {unit:8s} | {note}")


Suction Pressure         =  110.000 kPa      | Gauge pressure at pump inlet
Discharge Pressure       =  420.000 kPa      | Gauge pressure at pump discharge
Flow Rate                =  145.000 gpm      | Requested operating flow
Rated Flow               =  180.000 gpm      | Pump nameplate maximum for this exercise
Specific Gravity         =    1.000 ratio    | Water-like fluid
Pump Efficiency          =   72.000 %        | Operating efficiency estimate


## 5. Trace the Legacy Calculation Chain

This is the key investigation step.

Do not begin by asking whether the final answer is correct. Ask:

> **What unit enters and leaves each step?**

The chain should be understandable without relying on cell coordinates.


In [5]:
ws = wb["Calculation Chain"]

for row in range(4, 12):
    step = ws[f"A{row}"].value
    quantity = ws[f"B{row}"].value
    formula = ws[f"C{row}"].value
    value_or_formula = ws[f"D{row}"].value
    unit = ws[f"E{row}"].value
    comment = ws[f"F{row}"].value

    print(
        f"Step {step}: {quantity}\n"
        f"  stated source/formula: {formula}\n"
        f"  workbook formula/value: {value_or_formula}\n"
        f"  stated unit: {unit}\n"
        f"  note: {comment}\n"
    )


Step 1: Suction Pressure
  stated source/formula: Inputs!B4
  workbook formula/value: =Inputs!B4/6.89476
  stated unit: psi
  note: Converted from external kPa input

Step 2: Discharge Pressure
  stated source/formula: Inputs!B5
  workbook formula/value: =Inputs!B5/6.89476
  stated unit: psi
  note: Converted from external kPa input

Step 3: Differential Pressure
  stated source/formula: Discharge - Suction
  workbook formula/value: =(D5-D4)/6.89476
  stated unit: psi
  note: Feeds head calculation

Step 4: Pump Head
  stated source/formula: ΔP × 2.31 / SG
  workbook formula/value: =D6*2.31/Inputs!B8
  stated unit: ft
  note: Head from pressure difference

Step 5: Hydraulic Horsepower
  stated source/formula: Flow × Head × SG / 3960
  workbook formula/value: =Inputs!B6*D7*Inputs!B8/3960
  stated unit: hp
  note: Power delivered to fluid

Step 6: Efficiency Fraction
  stated source/formula: Efficiency % / 100
  workbook formula/value: =Inputs!B9/100
  stated unit: ratio
  note: Decimal 

## 6. Demonstrate the Duplicate-Conversion Defect

The workbook first converts **suction pressure** and **discharge pressure** from kPa to psi.

That means the pressure difference between those two converted values is **already in psi**.

The legacy chain then divides the pressure difference by the kPa-per-psi factor **again**. That is the Module 2 defect.

This demonstrates an important idea:

> **A correct conversion function can still be used incorrectly by the system.**


In [6]:
KPA_PER_PSI = 6.89476

suction_kpa = inputs["Suction Pressure"]
discharge_kpa = inputs["Discharge Pressure"]

suction_psi = suction_kpa / KPA_PER_PSI
discharge_psi = discharge_kpa / KPA_PER_PSI

correct_delta_psi = discharge_psi - suction_psi
legacy_delta_psi = correct_delta_psi / KPA_PER_PSI

print(f"Suction pressure:            {suction_psi:.6f} psi")
print(f"Discharge pressure:          {discharge_psi:.6f} psi")
print(f"CORRECT differential:        {correct_delta_psi:.6f} psi")
print(f"Legacy double-converted ΔP:  {legacy_delta_psi:.6f} psi")
print()
print(f"Legacy value is only {legacy_delta_psi/correct_delta_psi:.3%} of the correct differential.")


Suction pressure:            15.954145 psi
Discharge pressure:          60.915826 psi
CORRECT differential:        44.961681 psi
Legacy double-converted ΔP:  6.521138 psi

Legacy value is only 14.504% of the correct differential.


## 7. Build / Reuse the Unit Conversion Functions

These are the functions students should bring forward from Assignment 2.1.

For the demo we implement the pressure pair explicitly.


In [7]:
KPA_PER_PSI = 6.89476


def kpa_to_psi(kpa):
    '''Convert pressure from kilopascals to pounds per square inch.'''
    return kpa / KPA_PER_PSI


def psi_to_kpa(psi):
    '''Convert pressure from pounds per square inch to kilopascals.'''
    return psi * KPA_PER_PSI


print("6.89476 kPa =", kpa_to_psi(6.89476), "psi")
print("50 psi =", psi_to_kpa(50), "kPa")


6.89476 kPa = 1.0 psi
50 psi = 344.738 kPa


## 8. Verify the Unit Module Independently

A unit test proves the conversion function itself works.

It does **not** prove the larger calculation uses it correctly.


In [8]:
assert math.isclose(kpa_to_psi(6.89476), 1.0, rel_tol=1e-9)
assert math.isclose(psi_to_kpa(50), 344.738, rel_tol=1e-9)

original_kpa = 250.0
restored_kpa = psi_to_kpa(kpa_to_psi(original_kpa))
assert math.isclose(restored_kpa, original_kpa, rel_tol=1e-9)

print("Unit conversion checks passed.")


Unit conversion checks passed.


## 9. Review the Operating Limits

These limits are the source of truth for required validation behavior in this assignment.


In [9]:
ws = wb["Operating Limits"]

for row in range(4, 12):
    rule_id = ws[f"A{row}"].value
    quantity = ws[f"B{row}"].value
    condition = ws[f"C{row}"].value
    reason = ws[f"D{row}"].value
    behavior = ws[f"E{row}"].value

    print(f"{rule_id}: {quantity}")
    print(f"  condition: {condition}")
    print(f"  behavior:  {behavior}")
    print(f"  why:       {reason}\n")


L1: Suction pressure
  condition: >= 0 kPa
  behavior:  Reject
  why:       Negative gauge pressure is outside this exercise's supported model.

L2: Discharge pressure
  condition: >= 0 kPa
  behavior:  Reject
  why:       Negative gauge pressure is outside this exercise's supported model.

L3: Flow rate
  condition: > 0 gpm
  behavior:  Reject
  why:       The pump calculation assumes active forward flow.

L4: Rated flow
  condition: > 0 gpm
  behavior:  Reject
  why:       A non-positive rating is not meaningful.

L5: Specific gravity
  condition: > 0
  behavior:  Reject
  why:       Division by zero/nonphysical fluid property must be prevented.

L6: Pump efficiency
  condition: > 0 and <= 100%
  behavior:  Reject
  why:       Efficiency outside this range is unsupported.

R1: Discharge vs suction
  condition: Discharge pressure > suction pressure
  behavior:  Reject combination
  why:       This model assumes the pump adds pressure.

R2: Requested vs rated flow
  condition: Flow rat

## 10. Implement Complete Validation

Notice the two levels:

### Individual validation
A single value violates a documented limit.

### Combination validation
Each value can be legal by itself, but the relationship between values is not allowed.


In [10]:
def validate_pump_inputs(
    suction_pressure_kpa,
    discharge_pressure_kpa,
    flow_rate_gpm,
    rated_flow_gpm,
    specific_gravity,
    pump_efficiency_pct,
):
    # Individual validation
    if suction_pressure_kpa < 0:
        raise ValueError(
            f"suction_pressure_kpa must be >= 0; received {suction_pressure_kpa}"
        )

    if discharge_pressure_kpa < 0:
        raise ValueError(
            f"discharge_pressure_kpa must be >= 0; received {discharge_pressure_kpa}"
        )

    if flow_rate_gpm <= 0:
        raise ValueError(
            f"flow_rate_gpm must be > 0; received {flow_rate_gpm}"
        )

    if rated_flow_gpm <= 0:
        raise ValueError(
            f"rated_flow_gpm must be > 0; received {rated_flow_gpm}"
        )

    if specific_gravity <= 0:
        raise ValueError(
            f"specific_gravity must be > 0; received {specific_gravity}"
        )

    if pump_efficiency_pct <= 0 or pump_efficiency_pct > 100:
        raise ValueError(
            "pump_efficiency_pct must be > 0 and <= 100; "
            f"received {pump_efficiency_pct}"
        )

    # Combination / relational validation
    if discharge_pressure_kpa <= suction_pressure_kpa:
        raise ValueError(
            "discharge_pressure_kpa must be greater than suction_pressure_kpa"
        )

    if flow_rate_gpm > rated_flow_gpm:
        raise ValueError(
            "flow_rate_gpm cannot exceed rated_flow_gpm"
        )

    return True


print("Validation function ready.")


Validation function ready.


## 11. Demonstrate Individual Rejection


In [11]:
try:
    validate_pump_inputs(
        suction_pressure_kpa=-5,
        discharge_pressure_kpa=420,
        flow_rate_gpm=145,
        rated_flow_gpm=180,
        specific_gravity=1.0,
        pump_efficiency_pct=72,
    )
except ValueError as exc:
    print("REFUSED:", exc)


REFUSED: suction_pressure_kpa must be >= 0; received -5


## 12. Demonstrate Combination Rejection

Both pressures below are non-negative.

Individually they are legal.

Together they violate the rule that the pump model assumes discharge pressure is greater than suction pressure.


In [12]:
try:
    validate_pump_inputs(
        suction_pressure_kpa=420,
        discharge_pressure_kpa=110,
        flow_rate_gpm=145,
        rated_flow_gpm=180,
        specific_gravity=1.0,
        pump_efficiency_pct=72,
    )
except ValueError as exc:
    print("REFUSED:", exc)


REFUSED: discharge_pressure_kpa must be greater than suction_pressure_kpa


## 13. Build the Correct Multi-Step Calculation

The completed architecture is:

**Validate → Convert Once → Calculate in Known Units → Return Named Results**


In [13]:
FT_HEAD_PER_PSI_WATER = 2.31
HP_CONSTANT = 3960.0


def calculate_pump_performance(
    suction_pressure_kpa,
    discharge_pressure_kpa,
    flow_rate_gpm,
    rated_flow_gpm,
    specific_gravity,
    pump_efficiency_pct,
):
    # 1. Validate external inputs before calculation.
    validate_pump_inputs(
        suction_pressure_kpa,
        discharge_pressure_kpa,
        flow_rate_gpm,
        rated_flow_gpm,
        specific_gravity,
        pump_efficiency_pct,
    )

    # 2. Convert external pressure values ONCE at the input boundary.
    suction_pressure_psi = kpa_to_psi(suction_pressure_kpa)
    discharge_pressure_psi = kpa_to_psi(discharge_pressure_kpa)

    # 3. Named intermediate values in known units.
    differential_pressure_psi = discharge_pressure_psi - suction_pressure_psi

    pump_head_ft = (
        differential_pressure_psi
        * FT_HEAD_PER_PSI_WATER
        / specific_gravity
    )

    hydraulic_hp = (
        flow_rate_gpm
        * pump_head_ft
        * specific_gravity
        / HP_CONSTANT
    )

    efficiency_fraction = pump_efficiency_pct / 100.0
    brake_hp = hydraulic_hp / efficiency_fraction
    flow_margin_gpm = rated_flow_gpm - flow_rate_gpm

    # 4. Structured result makes intermediate values inspectable and testable.
    return {
        "suction_pressure_psi": suction_pressure_psi,
        "discharge_pressure_psi": discharge_pressure_psi,
        "differential_pressure_psi": differential_pressure_psi,
        "pump_head_ft": pump_head_ft,
        "hydraulic_hp": hydraulic_hp,
        "efficiency_fraction": efficiency_fraction,
        "brake_hp": brake_hp,
        "flow_margin_gpm": flow_margin_gpm,
    }


## 14. Run the Baseline Operating Case


In [14]:
result = calculate_pump_performance(
    suction_pressure_kpa=110,
    discharge_pressure_kpa=420,
    flow_rate_gpm=145,
    rated_flow_gpm=180,
    specific_gravity=1.0,
    pump_efficiency_pct=72,
)

for key, value in result.items():
    print(f"{key:28s}: {value:.6f}")


suction_pressure_psi        : 15.954145
discharge_pressure_psi      : 60.915826
differential_pressure_psi   : 44.961681
pump_head_ft                : 103.861483
hydraulic_hp                : 3.803009
efficiency_fraction         : 0.720000
brake_hp                    : 5.281957
flow_margin_gpm             : 35.000000


## 15. Compare Correct vs. Legacy Result

This is useful during the live demo because students can see how a single repeated conversion propagates through downstream calculations.


In [15]:
correct_head = result["pump_head_ft"]

legacy_head = legacy_delta_psi * FT_HEAD_PER_PSI_WATER / 1.0

print(f"Correct pump head: {correct_head:.6f} ft")
print(f"Legacy pump head:  {legacy_head:.6f} ft")
print(f"Difference:        {correct_head - legacy_head:.6f} ft")


Correct pump head: 103.861483 ft
Legacy pump head:  15.063829 ft
Difference:        88.797655 ft


## 16. Load the Independent Reference Cases

The workbook provides expected head and brake horsepower values that were generated independently of this function.

That distinction matters:

> **The function under test must not generate its own answer key.**


In [16]:
ws = wb["Reference Cases"]

reference_cases = []
for row in range(4, 7):
    reference_cases.append({
        "case": ws[f"A{row}"].value,
        "suction_kpa": ws[f"B{row}"].value,
        "discharge_kpa": ws[f"C{row}"].value,
        "flow_gpm": ws[f"D{row}"].value,
        "rated_gpm": ws[f"E{row}"].value,
        "sg": ws[f"F{row}"].value,
        "eff_pct": ws[f"G{row}"].value,
        "expected_head_ft": ws[f"H{row}"].value,
        "expected_brake_hp": ws[f"I{row}"].value,
    })

reference_cases


[{'case': 'RC-1',
  'suction_kpa': 110,
  'discharge_kpa': 420,
  'flow_gpm': 145,
  'rated_gpm': 180,
  'sg': 1,
  'eff_pct': 72,
  'expected_head_ft': 103.86148321333883,
  'expected_brake_hp': 5.281956743102599},
 {'case': 'RC-2',
  'suction_kpa': 95,
  'discharge_kpa': 360,
  'flow_gpm': 120,
  'rated_gpm': 160,
  'sg': 0.92,
  'eff_pct': 75,
  'expected_head_ft': 96.50523510355818,
  'expected_brake_hp': 3.5872653048595367},
 {'case': 'RC-3',
  'suction_kpa': 150,
  'discharge_kpa': 510,
  'flow_gpm': 150,
  'rated_gpm': 190,
  'sg': 1.1,
  'eff_pct': 68,
  'expected_head_ft': 109.64848667683866,
  'expected_brake_hp': 6.718657271865115}]

## 17. Verify Every Reference Case


In [17]:
for case in reference_cases:
    actual = calculate_pump_performance(
        case["suction_kpa"],
        case["discharge_kpa"],
        case["flow_gpm"],
        case["rated_gpm"],
        case["sg"],
        case["eff_pct"],
    )

    head_ok = math.isclose(
        actual["pump_head_ft"],
        case["expected_head_ft"],
        rel_tol=1e-9,
        abs_tol=1e-9,
    )
    bhp_ok = math.isclose(
        actual["brake_hp"],
        case["expected_brake_hp"],
        rel_tol=1e-9,
        abs_tol=1e-9,
    )

    print(
        case["case"],
        "| head:", "PASS" if head_ok else "FAIL",
        "| brake hp:", "PASS" if bhp_ok else "FAIL"
    )


RC-1 | head: PASS | brake hp: PASS
RC-2 | head: PASS | brake hp: PASS
RC-3 | head: PASS | brake hp: PASS


## 18. Build Demo Assertions

These simple assertions represent the logic that students later move into `pytest`.


In [18]:
# Known-correct calculations
rc1 = reference_cases[0]
r1 = calculate_pump_performance(
    rc1["suction_kpa"], rc1["discharge_kpa"], rc1["flow_gpm"],
    rc1["rated_gpm"], rc1["sg"], rc1["eff_pct"]
)
assert math.isclose(r1["pump_head_ft"], rc1["expected_head_ft"], rel_tol=1e-9)
assert math.isclose(r1["brake_hp"], rc1["expected_brake_hp"], rel_tol=1e-9)

# Boundary: flow exactly at rating is allowed.
r_boundary = calculate_pump_performance(
    100, 250, 50, 50, 1.0, 60
)
assert math.isclose(r_boundary["flow_margin_gpm"], 0.0, abs_tol=1e-12)

print("Known-correct and boundary assertions passed.")


Known-correct and boundary assertions passed.


## 19. Create a Small `raises()` Helper for the Demo

In the repository students should use `pytest.raises()`.

This helper lets the notebook demonstrate the same behavior without requiring the notebook itself to run under pytest.


In [19]:
def expect_value_error(fn, expected_text):
    try:
        fn()
    except ValueError as exc:
        message = str(exc)
        assert expected_text in message, (
            f"Expected '{expected_text}' in error message, got: {message}"
        )
        print("PASS — rejected with:", message)
        return
    raise AssertionError("Expected ValueError, but calculation succeeded.")


## 20. Prove Individual Rejections


In [20]:
expect_value_error(
    lambda: calculate_pump_performance(-1, 420, 145, 180, 1.0, 72),
    "suction_pressure_kpa",
)

expect_value_error(
    lambda: calculate_pump_performance(110, 420, 0, 180, 1.0, 72),
    "flow_rate_gpm",
)

expect_value_error(
    lambda: calculate_pump_performance(110, 420, 145, 180, 1.0, 0),
    "pump_efficiency_pct",
)


PASS — rejected with: suction_pressure_kpa must be >= 0; received -1
PASS — rejected with: flow_rate_gpm must be > 0; received 0
PASS — rejected with: pump_efficiency_pct must be > 0 and <= 100; received 0


## 21. Prove Combination Rejections


In [21]:
expect_value_error(
    lambda: calculate_pump_performance(420, 110, 145, 180, 1.0, 72),
    "discharge_pressure_kpa must be greater",
)

expect_value_error(
    lambda: calculate_pump_performance(110, 420, 200, 180, 1.0, 72),
    "flow_rate_gpm cannot exceed rated_flow_gpm",
)


PASS — rejected with: discharge_pressure_kpa must be greater than suction_pressure_kpa
PASS — rejected with: flow_rate_gpm cannot exceed rated_flow_gpm


## 22. Boundary Test Demonstration

Boundary tests should answer questions such as:

- Is suction pressure exactly 0 accepted?
- Is flow exactly equal to rated flow accepted?
- Is 100% efficiency accepted?
- What happens immediately outside the supported range?


In [22]:
# Accepted exact boundaries
accepted = calculate_pump_performance(
    suction_pressure_kpa=0,
    discharge_pressure_kpa=150,
    flow_rate_gpm=1,
    rated_flow_gpm=1,
    specific_gravity=0.85,
    pump_efficiency_pct=100,
)
print("Accepted exact boundary case.")
print("Flow margin:", accepted["flow_margin_gpm"])

# Just outside a supported boundary
expect_value_error(
    lambda: calculate_pump_performance(0, 150, 1, 1, 0.85, 100.001),
    "pump_efficiency_pct",
)


Accepted exact boundary case.
Flow margin: 0
PASS — rejected with: pump_efficiency_pct must be > 0 and <= 100; received 100.001


## 23. Unit-Boundary Integration Test

This test is especially important.

A unit test can prove `kpa_to_psi()` is correct.

An integration test proves the **full system uses it correctly**.


In [23]:
case = reference_cases[0]

actual = calculate_pump_performance(
    case["suction_kpa"],
    case["discharge_kpa"],
    case["flow_gpm"],
    case["rated_gpm"],
    case["sg"],
    case["eff_pct"],
)

expected_delta_psi = (
    kpa_to_psi(case["discharge_kpa"])
    - kpa_to_psi(case["suction_kpa"])
)

assert math.isclose(
    actual["differential_pressure_psi"],
    expected_delta_psi,
    rel_tol=1e-9,
)

print("PASS — pressure is converted once and the differential remains in psi.")


PASS — pressure is converted once and the differential remains in psi.


## 24. Demonstrate a Regression Test Catching Double Conversion

For teaching, intentionally recreate the bad behavior.

The assertion should fail against the buggy implementation.


In [24]:
def buggy_differential_pressure(suction_kpa, discharge_kpa):
    suction_psi = kpa_to_psi(suction_kpa)
    discharge_psi = kpa_to_psi(discharge_kpa)

    # BUG: pressure difference is already in psi.
    return (discharge_psi - suction_psi) / KPA_PER_PSI


correct_dp = (
    kpa_to_psi(420)
    - kpa_to_psi(110)
)
buggy_dp = buggy_differential_pressure(110, 420)

print("Correct ΔP:", correct_dp)
print("Buggy ΔP:  ", buggy_dp)
print("Would integration test pass?", math.isclose(buggy_dp, correct_dp, rel_tol=1e-9))


Correct ΔP: 44.96168104473542
Buggy ΔP:   6.52113794312426
Would integration test pass? False


## 25. Optional: Load Supplemental Validation Data

If you uploaded `PUMP_VALIDATION_DATA.xlsx`, the next cells classify and execute a wider set of cases.


In [26]:
SUPPLEMENTAL = Path("PUMP_VALIDATION_DATA.xlsx")

if SUPPLEMENTAL.exists():
    swb = load_workbook(SUPPLEMENTAL, data_only=True)
    print("Supplemental sheets:", swb.sheetnames)
else:
    swb = None
    print("Supplemental workbook not uploaded — skipping this optional section.")


Supplemental sheets: ['Read Me', 'Operating Runs', 'Validation Challenges', 'Boundary Cases']


## 26. Run the 30 Normal Operating Records


In [27]:
if swb is not None:
    ws = swb["Operating Runs"]
    passed = 0

    for row in range(4, 34):
        values = [ws.cell(row=row, column=c).value for c in range(1, 8)]
        run_id, suction, discharge, flow, rated, sg, eff = values

        calculate_pump_performance(
            suction, discharge, flow, rated, sg, eff
        )
        passed += 1

    print(f"{passed} normal operating records calculated successfully.")


30 normal operating records calculated successfully.


## 27. Classify the Validation Challenge Dataset

This is a useful classroom activity:

1. Have students predict the classification first.
2. Then execute the validator.
3. Compare prediction with actual behavior.


In [28]:
def classify_case(suction, discharge, flow, rated, sg, eff):
    try:
        calculate_pump_performance(
            suction, discharge, flow, rated, sg, eff
        )
        return "Calculate"
    except ValueError as exc:
        msg = str(exc)

        combination_markers = [
            "must be greater than suction_pressure_kpa",
            "cannot exceed rated_flow_gpm",
        ]

        if any(marker in msg for marker in combination_markers):
            return "Reject combination"
        return "Reject individual"


if swb is not None:
    ws = swb["Validation Challenges"]

    for row in range(4, 16):
        case_id = ws[f"A{row}"].value
        suction = ws[f"B{row}"].value
        discharge = ws[f"C{row}"].value
        flow = ws[f"D{row}"].value
        rated = ws[f"E{row}"].value
        sg = ws[f"F{row}"].value
        eff = ws[f"G{row}"].value

        print(
            case_id,
            "->",
            classify_case(suction, discharge, flow, rated, sg, eff)
        )


VC-01 -> Calculate
VC-02 -> Reject individual
VC-03 -> Reject individual
VC-04 -> Reject individual
VC-05 -> Reject individual
VC-06 -> Reject individual
VC-07 -> Reject individual
VC-08 -> Reject individual
VC-09 -> Reject combination
VC-10 -> Reject combination
VC-11 -> Reject combination
VC-12 -> Calculate


## 28. Run the Boundary Dataset


In [29]:
if swb is not None:
    ws = swb["Boundary Cases"]

    for row in range(4, 14):
        case_id = ws[f"A{row}"].value
        suction = ws[f"B{row}"].value
        discharge = ws[f"C{row}"].value
        flow = ws[f"D{row}"].value
        rated = ws[f"E{row}"].value
        sg = ws[f"F{row}"].value
        eff = ws[f"G{row}"].value
        intent = ws[f"H{row}"].value

        classification = classify_case(
            suction, discharge, flow, rated, sg, eff
        )

        print(f"{case_id}: {classification:20s} | {intent}")


BC-01: Calculate            | Exact accepted boundaries
BC-02: Reject individual    | Just below suction lower limit
BC-03: Calculate            | Very small positive flow
BC-04: Reject individual    | Exact disallowed flow boundary
BC-05: Calculate            | Barely positive pressure differential; flow at rating
BC-06: Reject combination   | Exact invalid pressure relationship
BC-07: Reject combination   | Just above rated flow
BC-08: Calculate            | Flow exactly at rating
BC-09: Calculate            | Very small positive specific gravity
BC-10: Reject individual    | Just above efficiency maximum


## 29. Optional Batch Results Table

This is a useful demonstration of why returning named intermediate values matters.


In [30]:
if swb is not None:
    ws = swb["Operating Runs"]
    demo_results = []

    for row in range(4, 9):  # first five records for display
        run_id = ws[f"A{row}"].value
        suction = ws[f"B{row}"].value
        discharge = ws[f"C{row}"].value
        flow = ws[f"D{row}"].value
        rated = ws[f"E{row}"].value
        sg = ws[f"F{row}"].value
        eff = ws[f"G{row}"].value

        result = calculate_pump_performance(
            suction, discharge, flow, rated, sg, eff
        )

        demo_results.append({
            "run_id": run_id,
            "head_ft": result["pump_head_ft"],
            "brake_hp": result["brake_hp"],
            "flow_margin_gpm": result["flow_margin_gpm"],
        })

    for row in demo_results:
        print(
            f"{row['run_id']}: "
            f"head={row['head_ft']:.2f} ft, "
            f"brake_hp={row['brake_hp']:.3f}, "
            f"flow_margin={row['flow_margin_gpm']:.1f} gpm"
        )


RUN-001: head=43.06 ft, brake_hp=1.638, flow_margin=82.2 gpm
RUN-002: head=77.64 ft, brake_hp=4.484, flow_margin=62.0 gpm
RUN-003: head=97.98 ft, brake_hp=9.545, flow_margin=30.0 gpm
RUN-004: head=127.81 ft, brake_hp=5.858, flow_margin=18.7 gpm
RUN-005: head=128.70 ft, brake_hp=4.065, flow_margin=107.3 gpm


## 30. AI Adversarial Testing Demonstration

A useful AI prompt for this assignment is:

> **Act as an adversarial software tester. Do not rewrite the implementation yet. Identify cases where each input is individually within its legal range but the combination violates the physical assumptions of the system. Produce test cases first and explain the relationship each case challenges.**

Then compare the suggestions against the workbook's documented operating limits.

The lesson is:

> **AI may suggest tests, but the engineering specification decides what is valid.**


## 31. Repository Structure for the Final Assignment

Students should move the completed work into:

```text
EGN321-Module2/
├── README.md
├── AI_LOG.md
├── requirements.txt
├── src/
│   ├── __init__.py
│   ├── units.py
│   ├── validation.py
│   └── calculation.py
└── tests/
    ├── test_units.py
    ├── test_validation.py
    └── test_calculation.py
```

Minimum new Assignment 2.2 evidence:

- 2 known-correct calculation tests
- 2 individual rejection tests
- 1 combination rejection test
- 1 boundary test
- 1 unit-boundary integration test
- Assignment 2.1 unit tests still passing


# Demo Wrap-Up

Ask students these questions before ending:

1. Where does pressure conversion occur?
2. What unit is `differential_pressure_psi` in?
3. Which test catches a repeated conversion?
4. Why can two valid values still form an invalid system state?
5. What is the difference between a unit test and an integration test?
6. Why can refusing to calculate be the correct engineering result?

### Core takeaway

**Correct component + incorrect integration = incorrect system.**

A trustworthy tool needs:

**validation + explicit units + named intermediate values + tests + refusal behavior.**
